# Xenofobia en redes sociales durante la pandemia de COVID-19 en Argentina


**Autora:** Natalia Debandi  y Anahi Gonzalez

**Fecha:** Marzo 2026  

Se parte de trabajar con un dataset y clasificador realizado en el marco del proyecto PIUBAMAS

### Fuente: Proyecto PIUBAMAS / HuggingFace

El dataset utilizado es `tweets_medios_arg.csv`, generado a partir del conjunto de datos público disponible en HuggingFace:

> **Dataset:** [`piubamas/articles_and_comments`](https://huggingface.co/datasets/piubamas/articles_and_comments)  

El dataset se encuentra etiquetado sobre discursos discriminatorios y de odio con este etiquetador:

> **Clasificador:** [`piuba-bigdata/beto-contextualized-hate-speech`](https://huggingface.co/piuba-bigdata/beto-contextualized-hate-speech)

Las categorías de odio etiquetadas son:
| Etiqueta | Descripción |
|----------|-------------|
| `CALLS` | Llamados a la acción violenta |
| `WOMEN` | Odio hacia mujeres |
| `LGBTI` | Odio hacia personas LGBTI+ |
| `RACISM` | Racismo / xenofobia |
| `CLASS` | Odio por clase social |
| `POLITICS` | Odio político |
| `DISABLED` | Odio hacia personas con discapacidad |
| `APPEARANCE` | Odio por apariencia física |
| `CRIMINAL` | Estigmatización criminal |


Los medios argentinos incluidos son:

| Usuario Twitter | Medio |
|-----------------|-------|
| `clarincom` | Clarín |
| `LANACION` | La Nación |
| `infobae` | Infobae |
| `pagina12` | Página 12 |
| `perfilcom` | Perfil |
| `cronica` | Crónica |
| `izquierdadiario` | La Izquierda Diario |
| `laderechadiario` | La Derecha Diario |
| `laderechamedios` | La Derecha Medios |





In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de visualización
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

## 2. Carga del dataset

In [2]:
df = pd.read_csv('data/tweets_medios_arg.csv', parse_dates=['date_tweet'])

print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df.head()

Dataset cargado: 5,493,397 filas × 18 columnas


,id,tweet_id_noticia,title,resumen,medio,date_tweet,text,user_id,HATEFUL,CALLS,WOMEN,LGBTI,RACISM,CLASS,POLITICS,DISABLED,APPEARANCE,CRIMINAL
0,0,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:03:00.900,@clarincom A mi me preocupa el trabajo.. La ev...,1532596098,0,0,0,0,0,0,0,0,0,0
1,1,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:05:04.500,@clarincom Lo que preocupa. https://t.co/Vmf9V...,1222094021687943168,0,0,0,0,0,0,0,0,0,0
2,2,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:06:03.100,@clarincom Lo que les preocupa. https://t.co/P...,1222094021687943168,0,0,0,0,0,0,0,0,0,0
3,3,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:11:02.300,@clarincom Le recomendaríamos al presidente de...,582286194,0,0,0,0,0,0,0,0,0,0
4,4,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:26:00.600,@clarincom Para salvar al correo de la quiebra...,951552761988034560,0,0,0,0,0,0,0,0,0,0


## 3. Descripción del dataset

### 3.1 Estructura de columnas

In [3]:
# Descripción de columnas y tipos de datos
col_desc = {
    'id': 'Índice del comentario dentro de la noticia',
    'tweet_id_noticia': 'ID del tweet de la noticia a la que responde el comentario',
    'title': 'Título de la noticia',
    'resumen': 'Texto del tweet de la noticia (resumen)',
    'medio': 'Cuenta de Twitter del medio de comunicación',
    'date_tweet': 'Fecha y hora del comentario (UTC)',
    'text': 'Texto del comentario',
    'user_id': 'ID del usuario que publicó el comentario',
    'HATEFUL': 'Indicador agregado: 1 si el comentario fue clasificado como odioso en alguna categoría',
    'CALLS': 'Llamados a la violencia',
    'WOMEN': 'Odio hacia mujeres',
    'LGBTI': 'Odio hacia LGBTI+',
    'RACISM': 'Racismo / xenofobia',
    'CLASS': 'Odio por clase social',
    'POLITICS': 'Odio político',
    'DISABLED': 'Odio hacia personas con discapacidad',
    'APPEARANCE': 'Odio por apariencia física',
    'CRIMINAL': 'Estigmatización criminal',
}

desc_df = pd.DataFrame({
    'Columna': col_desc.keys(),
    'Descripción': col_desc.values(),
    'Tipo': [df[c].dtype for c in col_desc.keys()]
})
display(desc_df)

,Columna,Descripción,Tipo
0,id,Índice del comentario dentro de la noticia,int64
1,tweet_id_noticia,ID del tweet de la noticia a la que responde e...,int64
2,title,Título de la noticia,str
3,resumen,Texto del tweet de la noticia (resumen),str
4,medio,Cuenta de Twitter del medio de comunicación,str
5,date_tweet,Fecha y hora del comentario (UTC),datetime64[us]
6,text,Texto del comentario,str
7,user_id,ID del usuario que publicó el comentario,int64
8,HATEFUL,Indicador agregado: 1 si el comentario fue cla...,int64
9,CALLS,Llamados a la violencia,int64


### 3.2 Estadísticas generales

In [4]:
print("=" * 55)
print("ESTADÍSTICAS GENERALES DEL DATASET")
print("=" * 55)
print(f"  Total de comentarios:      {len(df):>12,}")
print(f"  Comentarios odiosos:       {df['HATEFUL'].sum():>12,}  ({df['HATEFUL'].mean()*100:.1f}%)")
print(f"  Comentarios no odiosos:    {(df['HATEFUL']==0).sum():>12,}  ({(df['HATEFUL']==0).mean()*100:.1f}%)")
print(f"  Rango temporal:            {df['date_tweet'].min().date()} → {df['date_tweet'].max().date()}")
print(f"  Medios cubiertos:          {df['medio'].nunique():>12,}")
print(f"  Noticias únicas:           {df['tweet_id_noticia'].nunique():>12,}")
print(f"  Usuarios únicos:           {df['user_id'].nunique():>12,}")

ESTADÍSTICAS GENERALES DEL DATASET
  Total de comentarios:         5,493,397
  Comentarios odiosos:            340,593  (6.2%)
  Comentarios no odiosos:       5,152,804  (93.8%)
  Rango temporal:            2020-02-10 → 2021-06-25
  Medios cubiertos:                     9
  Noticias únicas:                330,600
  Usuarios únicos:                402,870


Se define un recorte entre el 1 de marzo de 2020 y el 15 de junio de 2021

### 3.3 Distribución por medio de comunicación

In [5]:
medios_stats = df.groupby('medio').agg(
    comentarios=('text', 'count'),
    odiosos=('HATEFUL', 'sum'),
    noticias=('tweet_id_noticia', 'nunique'),
    primer_tweet=('date_tweet', 'min'),
    ultimo_tweet=('date_tweet', 'max')
).sort_values('comentarios', ascending=False)

medios_stats['% odio'] = (medios_stats['odiosos'] / medios_stats['comentarios'] * 100).round(2)
medios_stats['primer_tweet'] = medios_stats['primer_tweet'].dt.date
medios_stats['ultimo_tweet'] = medios_stats['ultimo_tweet'].dt.date

display(medios_stats[['comentarios', 'noticias', 'odiosos', '% odio', 'primer_tweet', 'ultimo_tweet']])

,comentarios,noticias,odiosos,% odio,primer_tweet,ultimo_tweet
medio,,,,,,
infobae,1691830,98751,105128,6.21,2020-02-10,2021-06-25
LANACION,1574297,51536,94459,6.00,2020-02-10,2021-06-25
clarincom,1486018,66267,96918,6.52,2020-02-10,2021-06-25
pagina12,221971,29184,10242,4.61,2020-02-13,2021-06-25
cronica,205588,50316,15371,7.48,2020-02-10,2021-06-24
perfilcom,159860,21056,7237,4.53,2020-02-10,2021-06-25
laderechamedios,69944,2800,4726,6.76,2020-07-14,2021-02-26
laderechadiario,65907,2904,4869,7.39,2020-02-11,2021-06-24
izquierdadiario,17982,7786,1643,9.14,2020-03-06,2021-06-22


## 4. Filtrado por palabras clave: DNU, vacunas y migración

Se prepara un dataset para focalizar el tratamiento de la migracion en la pandemia, alrededor de dos temas: vacunación, medidas de aislamiento (a partir de los distintos DNU) y la derogación del DNU 70/17. Para eso se filtra por 3 palabras


| Palabra clave | Justificación |
|---|---|
| `DNU` | Decretos de necesidad y urgencia emitidos a lo largo de la pandemia |
| `extranjeros` | Término central del decreto y del debate público |
| `migrantes` | Término usado para reivindicar la población extranjera - no peyorativo |


Para complementar se incorporan palabras adicionales 
 'antecedentes','bolivianos', 'chino','peruanos', 'venezolanos', 'paraguayos', 'colombianos', 'chilenos', 'uruguayos', 'Covid','cuarentena', 'pandemia', 'coronavirus', 'covid-19', 'covid19', 'corona virus','cuarentena', 'pandemia', 'coronavirus', 'covid-19', 'covid19', 'corona virus','cuarentena', 'pandemia', 'coronavirus', 'covid-19', 'covid19', 'corona virus', 'expulsion', 'DNU 70/17'

La búsqueda es **insensible a mayúsculas/minúsculas** y no requiere coincidencia exacta de acentos (se normaliza antes de buscar). Se busca en los tres campos simultáneamente con `OR`.

In [6]:
import unicodedata

def normalizar(texto):
    """Minúsculas y sin tildes para búsqueda robusta."""
    if not isinstance(texto, str):
        return ''
    texto = texto.lower()
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    return texto

KEYWORDS = ['extranjeros', 'migrantes', 'extranjero', 'migrante', 'DNU 70/17','expulsión', 'DNU', 'antecedentes','bolivianos', 'chino','peruanos', 'venezolanos', 'paraguayos', 'colombianos', 'chilenos', 'uruguayos', 'Covid','cuarentena', 'pandemia', 'coronavirus', 'covid-19', 'covid19', 'corona virus','cuarentena', 'pandemia', 'coronavirus', 'covid-19', 'covid19', 'corona virus','cuarentena', 'pandemia', 'coronavirus', 'covid-19', 'covid19', 'corona virus']

# Normalizar los tres campos de texto
for col in ['title', 'resumen', 'text']:
    df[f'_{col}_norm'] = df[col].apply(normalizar)

# Máscara: al menos una keyword en alguno de los tres campos
mascara = df[['_title_norm', '_resumen_norm', '_text_norm']].apply(
    lambda col: col.str.contains('|'.join(KEYWORDS), regex=True)
).any(axis=1)

df_dnu = df[mascara].drop(columns=['_title_norm', '_resumen_norm', '_text_norm']).copy()

# Limpiar columnas temporales del df original
df.drop(columns=['_title_norm', '_resumen_norm', '_text_norm'], inplace=True)

print(f"Filas totales:          {len(df):>10,}")
print(f"Filas filtradas (DNU):  {len(df_dnu):>10,}  ({len(df_dnu)/len(df)*100:.2f}%)")
print()
print("Distribución por medio:")
display(
    df_dnu.groupby('medio').agg(
        comentarios=('text', 'count'),
        odiosos=('HATEFUL', 'sum')
    ).assign(**{'% odio': lambda x: (x['odiosos'] / x['comentarios'] * 100).round(2)})
    .sort_values('comentarios', ascending=False)
)

# Guardar subconjunto
df_dnu.to_csv('data/piubamas_dnu_v2.csv', index=False)
print("\nGuardado en: data/piubamas_dnu_v2.csv")

Filas totales:           5,493,397
Filas filtradas (DNU):   1,056,507  (19.23%)

Distribución por medio:


,comentarios,odiosos,% odio
medio,,,
infobae,334801,18799,5.61
clarincom,300900,17687,5.88
LANACION,279580,15752,5.63
cronica,48141,2590,5.38
pagina12,36897,1740,4.72
perfilcom,32926,1227,3.73
laderechadiario,12878,1028,7.98
laderechamedios,8080,649,8.03
izquierdadiario,2304,202,8.77



Guardado en: data/piubamas_dnu_v2.csv
